# NUFROST Ablation Study (Colab)

This notebook runs ablation experiments for the NUFROST algorithm to quantify the contribution of each key component.

**Fixed evaluation scenarios:**
- Random‑point masking (40 % of valid observations removed).
- Continuous‑gap simulation (120‑day artificial gap).

**Ablation variants:**
1. Full NUFROST (default config).
2. w/o preferred frequencies (`frequency_selection="spectral"`).
3. w/o parabolic refinement (`refine_peaks=False`).
4. w/o Huber robust fitting (`huber_iters=0`).
5. w/o frequency‑weighted ridge (`freq_weight=0.0`).
6. w/o linear trend (`include_trend=False`).

The notebook auto‑discovers all (band, lon, lat) cubes in the data directory, evaluates each variant on the two scenarios, and appends results incrementally to a CSV file.
Resume‑from‑previous is supported by checking the output CSV.

## 1. Configuration

In [1]:
from pathlib import Path

MOUNT_POINT_IN_COLAB = Path("/content/drive")
PROJECT_PATH_IN_GDRIVE = Path("WorkSpaces/nufrost")
PROJECT_DIR = MOUNT_POINT_IN_COLAB / "MyDrive" / PROJECT_PATH_IN_GDRIVE
IMAGE_DIR   = PROJECT_DIR / "data/hls"          # or "data/sentinel-2"
OUTPUT_DIR  = PROJECT_DIR / "data/output"
CACHE_DIR   = PROJECT_DIR / "data/cache/colab"
OUTPUT_CSV_PATH = OUTPUT_DIR / "ablation_results.csv"

# If you want to limit to specific files, list them here; leave empty to auto‑discover.
IMAGE_NAMES = []

# Fixed evaluation settings
RANDOM_POINTS_NUM = 50000          # total random points to mask (for sparse scenario)
RANDOM_MASK_RATIO = 0.4            # 40 % of valid observations will be removed
CONTINUOUS_GAP_DAYS = 120          # 120‑day artificial gap

# Random seed for reproducibility (will be used per image chunk)
BASE_SEED = 42

# Parallel jobs (-1 uses all available cores)
N_JOBS = -1


## 2. Mount Google Drive

In [2]:
import os
from google.colab import drive # type: ignore[import]

drive.mount(MOUNT_POINT_IN_COLAB.as_posix())
os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")


## 3. Install Dependencies

In [3]:
!apt-get install -y gdal-bin
%pip install -r requirements.txt


## 4. Import Modules & Define Ablation Configurations

In [4]:
import src.data_loader
import importlib
import src.evaluation
import pandas as pd
import glob
import re
from collections import defaultdict
import numpy as np

from config import build_args
from IPython.display import display

importlib.reload(src.nufrost)
importlib.reload(src.zhu2015)
importlib.reload(src.hants)
importlib.reload(src.evaluation)
importlib.reload(src.data_loader)

# Ablation definitions: each entry is a dict of parameter overrides
ABLATION_VARIANTS = [
    {
        "name": "Full NUFROST",
        "algorithm": "NuFrost",
        "overrides": {},
    },
    {
        "name": "w/o preferred frequencies",
        "algorithm": "NuFrost",
        "overrides": {"frequency_selection": "spectral"},
    },
    {
        "name": "w/o parabolic refinement",
        "algorithm": "NuFrost",
        "overrides": {"refine_peaks": False},
    },
    {
        "name": "w/o Huber robust fitting",
        "algorithm": "NuFrost",
        "overrides": {"huber_iters": 0},
    },
    {
        "name": "w/o frequency‑weighted ridge",
        "algorithm": "NuFrost",
        "overrides": {"freq_weight": 0.0},
    },
    {
        "name": "w/o linear trend",
        "algorithm": "NuFrost",
        "overrides": {"include_trend": False},
    },
    # Optionally add baseline methods for reference
    {
        "name": "Zhu2015",
        "algorithm": "Zhu2015",
        "overrides": {},
    },
    {
        "name": "HANTS",
        "algorithm": "HANTS",
        "overrides": {},
    },
]


## 5. Auto‑discover Image Chunks

In [5]:
if IMAGE_NAMES:
    image_paths_list = [[(IMAGE_DIR / name).as_posix()] for name in IMAGE_NAMES]
else:
    # Auto‑detect all distinct coordinates and bands, then construct their VRTs
    files = glob.glob((IMAGE_DIR / "*.tif").as_posix())
    loc_ids = set()
    for f in files:
        f_name = Path(f).name
        match = re.search(r"_([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+).*?(?:_part\d+)?(?:-\d{10}-\d{10})?\.tif$", f_name)
        if match:
            band = match.group(1)
            lon = float(match.group(2))
            lat = float(match.group(3))
            loc_ids.add((band, lon, lat))

    image_paths_list = []
    for band, lon, lat in loc_ids:
        # find_image_chunks returns the ordered VRT paths for one coordinate/band
        chunks = src.data_loader.find_image_chunks(IMAGE_DIR.as_posix(), lon, lat, band, cache_dir=CACHE_DIR.as_posix())
        if chunks:
            image_paths_list.append(chunks)

print(f"Found {len(image_paths_list)} distinct spatial/band chunks to evaluate.")
if not image_paths_list:
    print("No image chunks found. Please check IMAGE_DIR and filename patterns.")


## 6. Run Ablation Experiments

In [6]:
import pandas as pd
import os
from pathlib import Path
import time

# Ensure output directory exists
Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

# Load previously evaluated results (if any) to resume
evaluated = set()
if OUTPUT_CSV_PATH.exists():
    try:
        existing_df = pd.read_csv(OUTPUT_CSV_PATH)
        for _, row in existing_df.iterrows():
            evaluated.add((row["Image"], row["Variant"], row["Scenario"]))
        print(f"Found existing results for {len(existing_df)} rows. Resuming...")
    except Exception as e:
        print(f"Could not read existing CSV: {e}")

all_results = []

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    match = re.search(r"([A-Z0-9]+_lon[0-9.]+_lat[0-9.]+)", first_path.stem)
    loc_id = match.group(1) if match else first_path.stem

    print(f"\n--- Evaluating: {loc_id} ---")

    for variant in ABLATION_VARIANTS:
        variant_name = variant["name"]
        algorithm = variant["algorithm"]
        overrides = variant["overrides"]

        # Determine which scenario(s) to run for this variant
        # For baseline methods (Zhu2015, HANTS) we only run random‑point masking
        # because they are not designed for the continuous‑gap scenario.
        scenarios = ["random", "gap"]
        if algorithm in ("Zhu2015", "HANTS"):
            scenarios = ["random"]

        for scenario in scenarios:
            if (loc_id, variant_name, scenario) in evaluated:
                print(f"    Skipping {variant_name} – {scenario} (already evaluated)")
                continue

            print(f"    Running {variant_name} – {scenario}")

            # Build args with appropriate overrides
            args = build_args({})
            args.image = image_paths
            args.cache_dir = CACHE_DIR.as_posix()
            args.n_jobs = N_JOBS
            args.force_refresh = False

            # Apply ablation overrides (only for NuFrost variants)
            for k, v in overrides.items():
                setattr(args, k, v)

            # Seed for reproducibility (different per image/variant/scenario)
            seed = BASE_SEED + hash(loc_id) % 1000 + hash(variant_name) % 1000 + hash(scenario) % 1000
            np.random.seed(seed)

            start_time = time.time()
            if scenario == "random":
                # Random‑point masking with a fixed number of points
                # The masking ratio is implicitly controlled by num_points.
                # We'll approximate the ratio by using a fixed large number of points.
                df_results = src.evaluation.evaluate_algorithms(
                    image_path=args.image,
                    args=args,
                    num_points=RANDOM_POINTS_NUM,
                    n_jobs=args.n_jobs,
                )
            else:  # "gap"
                df_results = src.evaluation.evaluate_timeseries_comprehensive(
                    image_path=args.image,
                    args=args,
                    num_samples=RANDOM_POINTS_NUM,  # use same sample size
                    simulate_gap_days=CONTINUOUS_GAP_DAYS,
                    n_jobs=args.n_jobs,
                )

            elapsed = time.time() - start_time
            print(f"      finished in {elapsed:.1f}s")

            if df_results.empty:
                print(f"      WARNING: No results for {variant_name} – {scenario}")
                continue

            # Add metadata columns
            df_results["Image"] = loc_id
            df_results["Variant"] = variant_name
            df_results["Scenario"] = scenario
            df_results["Algorithm"] = algorithm
            df_results["Seed"] = seed

            all_results.append(df_results)

            # Incremental save after each variant/scenario
            Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
            header = not OUTPUT_CSV_PATH.exists()
            df_results.to_csv(OUTPUT_CSV_PATH, mode="a", header=header, index=False)

print("\n========== Ablation Evaluation Complete ==========")
print(f"Results saved to: {OUTPUT_CSV_PATH}")


## 7. Quick Summary (Optional)

In [ ]:
if OUTPUT_CSV_PATH.exists():
    df_all = pd.read_csv(OUTPUT_CSV_PATH)
    print(f"Total rows collected: {len(df_all)}")
    print("\nMean metrics per variant and scenario:")
    summary = df_all.groupby(["Variant", "Scenario"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    display(summary)
else:
    print("No results CSV found.")
